![earthkit-data-logo](https://github.com/ecmwf/logos/raw/refs/heads/main/logos/earthkit/earthkit-data-light.svg)

# Writing to targets

In this notebook you will see how to:

- write data into a target 
- use an encoder

First, we read some GRIB data from disk with [from_source](https://earthkit-data.readthedocs.io/en/latest/concepts/inputs/from_source.html) and load it into a fieldlist.

In [1]:
import earthkit.data as ekd

d = ekd.from_source("file", "../test4.grib")
fl = d.to_fieldlist()
len(fl)

4

### Targets

A [target](https://earthkit-data.readthedocs.io/en/latest/concepts/targets/index.html) can represent a file, a database, a remote server etc. Data is written/added to a target by using a suitable [encoder](https://earthkit-data.readthedocs.io/en/latest/concepts/encoders.html#encoders).
    
We use [to_target()](https://earthkit-data.readthedocs.io/en/latest/concepts/targets/to_target.html#to_target) to write the data into target. The encoder is automatically guessed from the input data (when possible) or from the output path's suffix.

In [2]:
# write all the data to a file
fl.to_target("file", "_my.grib")

# check the result
ekd.from_source("file", "_my.grib").to_fieldlist().ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,500,pressure,0,regular_ll
1,z,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,500,pressure,0,regular_ll
2,t,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,850,pressure,0,regular_ll
3,z,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,850,pressure,0,regular_ll


In [3]:
# write only a slice to target
fl[2:4].to_target("file", "_my.grib")

# check the result
ekd.from_source("file", "_my.grib").to_fieldlist().ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,850,pressure,0,regular_ll
1,z,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,850,pressure,0,regular_ll


### Target as a context manager

A target can be used as a context manager.

In [4]:
# write data to target
with ekd.create_target("file", "_my.grib") as t:
    t.write(fl)

# check the result
ekd.from_source("file", "_my.grib").to_fieldlist().ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,500,pressure,0,regular_ll
1,z,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,500,pressure,0,regular_ll
2,t,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,850,pressure,0,regular_ll
3,z,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,850,pressure,0,regular_ll


The code below write each field individually to the target.

In [5]:
# write data to target
with ekd.create_target("file", "_my.grib") as t:
    for f in fl:
        if f.parameter.variable() == "z":
            t.write(f)

# check the result
ekd.from_source("file", "_my.grib").to_fieldlist().ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,z,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,500,pressure,0,regular_ll
1,z,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,850,pressure,0,regular_ll


### Using an encoder

In the example below we specify extra GRIB metadata. The [GRIB encoder](https://earthkit-data.readthedocs.io/en/latest/how-tos/target/grib_encoder.html), which is used behind the scenes, will add this to the resulting file.

In [6]:
# write to target and set GRIB keys for the output
fl[:2].to_target("file", "_my.grib", metadata={"date": 20250108, "bitsPerValue": 8})

# check the result
ekd.from_source("file", "_my.grib").to_fieldlist().ls(keys=["time.valid_datetime", "time.base_datetime", "time.step", "metadata.bitsPerValue"])

,time.valid_datetime,time.base_datetime,time.step,metadata.bitsPerValue
0,2025-01-08 12:00:00,2025-01-08 12:00:00,0 days,8
1,2025-01-08 12:00:00,2025-01-08 12:00:00,0 days,8


We can do the same by creating an encoder object.

In [7]:
# create an encoder
encoder = ekd.create_encoder("grib", metadata={"date": 20250108})

# write to target
fl[:2].to_target("file", "_my.grib", encoder=encoder,  metadata={"bitsPerValue": 8})

# check the result
ekd.from_source("file", "_my.grib").to_fieldlist().ls(keys=["time.valid_datetime", "time.base_datetime", "time.step", "metadata.bitsPerValue"])

,time.valid_datetime,time.base_datetime,time.step,metadata.bitsPerValue
0,2025-01-08 12:00:00,2025-01-08 12:00:00,0 days,8
1,2025-01-08 12:00:00,2025-01-08 12:00:00,0 days,8


The code below is equivalent to the one in the previous cell.

In [8]:
# create an encoder
encoder = ekd.create_encoder("grib", metadata={"date": 20250108})

# write to target
with ekd.create_target("file", "_res_t.grib") as target:
    target.write(fl[:2], encoder=encoder,  metadata={"bitsPerValue": 8})

# check the result
ekd.from_source("file", "_my.grib").to_fieldlist().ls(keys=["time.valid_datetime", "time.base_datetime", "time.step", "metadata.bitsPerValue"])

,time.valid_datetime,time.base_datetime,time.step,metadata.bitsPerValue
0,2025-01-08 12:00:00,2025-01-08 12:00:00,0 days,8
1,2025-01-08 12:00:00,2025-01-08 12:00:00,0 days,8
